In [ ]:
!pip install --force-reinstall \
    numpy==1.24.4 \
    scipy==1.11.2 \
    scikit-learn==1.3.2 \
    torch torchvision torchaudio opencv-python pandas


In [1]:
import numpy as np
import scipy
import sklearn
print(np.__version__, scipy.__version__, sklearn.__version__)

1.24.4 1.11.2 1.3.2


In [2]:
# Clona il repo XDVioDet
!git clone https://github.com/Roc-Ng/XDVioDet.git
%cd XDVioDet
print("clone eseguito")

Cloning into 'XDVioDet'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 70 (delta 2), reused 2 (delta 2), pack-reused 62 (from 1)
Receiving objects: 100% (70/70), 3.48 MiB | 13.69 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/kaggle/working/XDVioDet
clone eseguito


In [3]:
!ls /kaggle/working/XDVioDet

ckpt	    infer.py   list	model.py   README.md  train.py
dataset.py  layers.py  main.py	option.py  test.py    utils.py


In [ ]:
# ============================================================
# Customization of the XDVioDet repository
# ------------------------------------------------------------
# The original repository is adapted by overwriting specific
# configuration and inference scripts.
# These modifications are required to:
# - match the feature format used in this work
# - ensure full experimental reproducibility
# ============================================================


In [4]:
%%writefile option.py
import argparse

# Argument parser for the XDVioDet framework.
# This file replaces the original option.py in order to define
# dataset-specific and modality-specific experimental settings.
parser = argparse.ArgumentParser(description='Weakly Supervised Anomaly Detection')

# Input modality configuration
parser.add_argument(
    '--modality',
    default='MIX2',
    help='Input modality type: AUDIO, RGB, FLOW, MIX1, MIX2, MIX3, or MIX_ALL'
)

# Training feature lists
parser.add_argument('--rgb-list', default='list/rgb.list', help='List of RGB feature files')
parser.add_argument('--flow-list', default='list/flow.list', help='List of optical flow feature files')
parser.add_argument('--audio-list', default='list/audio.list', help='List of audio feature files')

# Test feature lists
parser.add_argument('--test-rgb-list', default='list/rgb_test.list', help='List of test RGB feature files')
parser.add_argument('--test-flow-list', default='list/flow_test.list', help='List of test optical flow feature files')
parser.add_argument('--test-audio-list', default='list/audio_test.list', help='List of test audio feature files')

# Ground truth annotations (frame-level)
parser.add_argument('--gt', default='list/gt.npy', help='Ground truth labels file')

# Hardware configuration
parser.add_argument('--gpus', default=0, type=int, choices=[-1, 0, 1], help='GPU selection (-1 for CPU)')

# Optimization parameters
parser.add_argument('--lr', type=float, default=1e-4, help='Learning rate')
parser.add_argument('--batch-size', type=int, default=128, help='Batch size')
parser.add_argument('--workers', type=int, default=2, help='Number of DataLoader workers')

# Model configuration
parser.add_argument('--model-name', default='wsanodet', help='Model name used for saving checkpoints')
parser.add_argument('--pretrained-ckpt', default=None, help='Path to a pretrained checkpoint')
parser.add_argument(
    '--feature-size',
    type=int,
    default=1024 + 128,
    help='Feature dimensionality (RGB + Audio)'
)
parser.add_argument('--num-classes', type=int, default=1, help='Number of output classes')

# Dataset and training setup
parser.add_argument('--dataset-name', default='XD-Violence', help='Dataset name')
parser.add_argument('--max-seqlen', type=int, default=200, help='Maximum sequence length')
parser.add_argument('--max-epoch', type=int, default=50, help='Maximum number of training epochs')

Overwriting option.py


In [5]:
%%writefile test.py
from sklearn.metrics import auc, precision_recall_curve
import numpy as np
import torch

def test(dataloader, model, device, gt):
    """
    Perform model evaluation on the test set.

    This function computes video-level predictions and additionally
    saves frame-level predictions to disk in order to enable
    fine-grained post-processing and evaluation.
    """
    with torch.no_grad():
        model.eval()

        # Tensors for storing predictions
        pred = torch.zeros(0).to(device)
        pred2 = torch.zeros(0).to(device)

        for i, input in enumerate(dataloader):
            input = input.to(device)

            # Forward pass
            logits, logits2 = model(inputs=input, seq_len=None)

            # Offline detection branch
            logits = torch.squeeze(logits)
            sig = torch.sigmoid(logits)
            sig = torch.mean(sig, 0)
            pred = torch.cat((pred, sig))

            # Online detection branch
            logits2 = torch.squeeze(logits2)
            sig2 = torch.sigmoid(logits2)
            sig2 = torch.mean(sig2, 0)

            # Adjust dimensions (required for audio modality)
            sig2 = torch.unsqueeze(sig2, 1)
            pred2 = torch.cat((pred2, sig2))

        # Convert predictions to NumPy arrays
        pred = list(pred.cpu().detach().numpy())
        pred2 = list(pred2.cpu().detach().numpy())

        # ============================================================
        # Frame-level prediction saving (CUSTOM MODIFICATION)
        # ------------------------------------------------------------
        # Each clip-level prediction corresponds to 16 frames.
        # Predictions are therefore repeated to obtain frame-level
        # scores, which are saved for subsequent analysis.
        # ============================================================
        np.save("pred_framelevel.npy", np.repeat(pred, 16))
        np.save("pred2_framelevel.npy", np.repeat(pred2, 16))
        # ============================================================

        # Precision-Recall AUC (offline branch)
        precision, recall, _ = precision_recall_curve(list(gt), np.repeat(pred, 16))
        pr_auc = auc(recall, precision)

        # Precision-Recall AUC (online branch)
        precision, recall, _ = precision_recall_curve(list(gt), np.repeat(pred2, 16))
        pr_auc2 = auc(recall, precision)

        return pr_auc, pr_auc2

Overwriting test.py


In [6]:
%cd /kaggle/working/XDVioDet

!python infer.py \
    --modality MIX2 \
    --test-rgb-list /kaggle/input/xdv-features/list/list/rgb_test.list \
    --test-flow-list /kaggle/input/xdv-features/list/list/flow_test.list \
    --test-audio-list /kaggle/input/xdv-features/list/list/audio_test.list \
    --gt /kaggle/input/xdv-features/list/list/gt.npy \
    --max-seqlen 128 \
    --workers 0


/kaggle/working/XDVioDet
Time:58.32157921791077
offline pr_auc:0.79; online pr_auc:0.7433



In [7]:
def load_video_boundaries(list_file):
    """
    Parse a .list file and compute the number of clips for each video.

    Each video is split into multiple clips saved as separate feature files
    (e.g., video_name__0.npy, video_name__1.npy, ...).
    This function groups clips belonging to the same original video.

    Returns:
        A list of tuples (video_name, number_of_clips).
    """
    paths = sorted([l.strip() for l in open(list_file)])
    video_info = {}
    for p in paths: 
        fname = p.split("/")[-1] # Extract file name without path
        base = "__".join(fname.split("__")[:-1])  # Remove clip index suffix to obtain the base video name

        # Count how many clips belong to each video
        if base not in video_info:
            video_info[base] = 0
        video_info[base] += 1 
    return list(video_info.items()) 


def detect_fight_videos(list_file):
    """
    Identify videos containing fight events based on file naming conventions.

    In the XD-Violence dataset, video-level labels are encoded in the file name.
    A video is considered a fight video if label 'B1' is present.

    Returns:
        A dictionary {video_name: 1 or 0}, where:
        - 1 indicates a fight video
        - 0 indicates a non-fight video
    """
    paths = sorted([l.strip() for l in open(list_file)])
    fight_dict = {}
    for p in paths:
        fname = p.split("/")[-1]
        base = "__".join(fname.split("__")[:-1]) 
        if "_label_" in base: 
            # Extract labels from the file name
            after = base.split("_label_")[1]
            labels = after.split("-")
            # Check if the fight label 'B1' is present
            is_fight = any(lbl == "B1" for lbl in labels)
            fight_dict[base] = int(is_fight)
        else:
            fight_dict[base] = 0
    return fight_dict 


In [8]:
def get_fight_frames(pred, gt, list_file):
    """
    Build frame-level predictions and ground truth vectors focusing on fight videos.

    For fight videos, the original predictions and ground truth are retained.
    For non-fight videos, predictions and ground truth are set to zero in order
    to exclude them from the evaluation.

    Each clip corresponds to 16 frames.
    """
    video_info = load_video_boundaries(list_file)
    fight_dict = detect_fight_videos(list_file)

    frame_idx = 0
    fight_pred = []
    fight_gt = []

    for video_name, num_clips in video_info:
        frames = num_clips * 16
        if fight_dict[video_name] == 1: 
            # Retain frame-level predictions and ground truth for fight videos
            fight_pred.extend(pred[frame_idx: frame_idx + frames])
            fight_gt.extend(gt[frame_idx: frame_idx + frames])
        else:
            # Set predictions and ground truth to zero for non-fight videos
            fight_pred.extend([0] * frames)
            fight_gt.extend([0] * frames)
        frame_idx += frames

    return np.array(fight_pred), np.array(fight_gt)


In [12]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc as sk_auc
# Load frame-level predictions and ground truth
# Predictions are generated and saved during the inference stage
pred = np.load("pred_framelevel.npy")  
gt = np.load("list/gt.npy") 

# Extract frame-level predictions and labels for fight videos
fight_pred, fight_gt = get_fight_frames(pred, gt, "list/rgb_test.list")

# Compute Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(fight_gt, fight_pred)

# Compute F1-score for each threshold (with numerical stability term)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

# Select the threshold that maximizes the F1-score
best_idx = f1_scores.argmax()
best_f1 = f1_scores[best_idx]
best_precision = precision[best_idx]
best_recall = recall[best_idx]

print("Best F1:", best_f1)
print(" Best Precision:", best_precision)
print(" Best Recall:", best_recall)

# Compute area under the Precision-Recall curve (PR-AUC)
pr_auc = sk_auc(recall, precision)
print("PR-AUC_fights:", pr_auc)

roc_auc = roc_auc_score(fight_gt, fight_pred)
print("ROC-AUC_fights:", roc_auc)


Best F1: 0.8540469235837834
 Best Precision: 0.8662079510703364
 Best Recall: 0.8422226351979186
PR-AUC_fights: 0.8910307354670827
ROC-AUC_fights: 0.9903969354558604


In [10]:
def get_fight_videos_for_sub(pred, gt, list_file):
    """
    Extract frame-level predictions and ground truth for the computation
    of subset-based metrics (AUC_sub and AP_sub).

    Only fight videos containing at least one positive frame are considered.
    This is required to ensure that ROC-AUC and PR-AUC are well-defined.
    """
    video_info = load_video_boundaries(list_file)
    fight_dict = detect_fight_videos(list_file)

    frame_idx = 0

    sub_pred = []
    sub_gt = []

    for video_name, num_clips in video_info:
        frames = num_clips * 16

        # Consider only fight videos
        if fight_dict[video_name] == 1:

            vid_pred = pred[frame_idx: frame_idx + frames]
            vid_gt   = gt[frame_idx: frame_idx + frames]

            # AUC_sub requires at least one positive frame in the video
            if np.sum(vid_gt) > 0:
                sub_pred.extend(vid_pred)
                sub_gt.extend(vid_gt)

        frame_idx += frames

    return np.array(sub_pred), np.array(sub_gt)


In [13]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve

# Extract frame-level predictions and ground truth
# restricted to fight videos with at least one positive frame
fight_pred_sub, fight_gt_sub = get_fight_videos_for_sub(
    pred, gt, "list/rgb_test.list"
)

# Compute subset-based ROC-AUC (AUC_sub)
roc_auc_sub = roc_auc_score(fight_gt_sub, fight_pred_sub)
print("ROC-AUC_sub (fight videos only):", roc_auc_sub)

# Compute subset-based Average Precision (AP_sub)
precision_sub, recall_sub, _ = precision_recall_curve(fight_gt_sub, fight_pred_sub)
pr_auc_sub = auc(recall_sub, precision_sub)
print("PR-AUC_sub (fight videos only):", pr_auc_sub)


ROC-AUC_sub (fight videos only): 0.8236414723129616
PR-AUC_sub (fight videos only): 0.9592757445353011
